# 65 — Documentary Evidence Fusion

Combines emissions summaries, CEMS evidence and catalog coverage into a
documentary evidence score per site-year.

In [ ]:
import os, json, hashlib, platform, sys, re
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
DATA = PROJECT_ROOT / "data_sources"
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        if path.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(path), "status": "empty_file"}
        df = pd.read_csv(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def safe_read_parquet(path: Path):
    if not path.exists():
        return pd.DataFrame(), {"path": str(path), "status": "missing"}
    try:
        df = pd.read_parquet(path)
        if df.empty:
            return pd.DataFrame(), {"path": str(path), "status": "no_rows", "sha256": sha256_file(path)}
        return df, {"path": str(path), "status": "ok", "sha256": sha256_file(path), "rows": int(len(df))}
    except Exception as e:
        return pd.DataFrame(), {"path": str(path), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def add_artifact(manifest: dict, p: Path):
    if p.exists():
        manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})

def slugify(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

def standardise_pollutant(x):
    s = str(x).strip().lower()
    mapping = {
        "oxides of nitrogen": "NOx",
        "nox": "NOx",
        "no2": "NO2",
        "no": "NO",
        "particulates": "Particulates",
        "dust": "Particulates",
        "pm10": "PM10",
        "pm2.5": "PM2.5",
        "sulphur dioxide": "SO2",
        "so2": "SO2",
        "hydrogen chloride": "HCl",
        "hcl": "HCl",
        "total organic carbon": "TOC",
        "toc": "TOC",
        "carbon monoxide": "CO",
        "co": "CO",
        "ammonia": "NH3",
        "nh3": "NH3",
    }
    return mapping.get(s, str(x).strip())

In [ ]:
step = "65_documentary_evidence_fusion"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

linked_emissions, _ = safe_read_csv(OUTPUTS / "64_documentary_site_year_linkage" / "linked_emissions.csv")
linked_cems, _ = safe_read_csv(OUTPUTS / "64_documentary_site_year_linkage" / "linked_cems.csv")
linked_catalog, _ = safe_read_csv(OUTPUTS / "64_documentary_site_year_linkage" / "linked_catalog.csv")

years = set()
if not linked_emissions.empty and "report_year" in linked_emissions.columns:
    years |= set(pd.to_numeric(linked_emissions["report_year"], errors="coerce").dropna().astype(int).tolist())
if not linked_cems.empty and "report_year" in linked_cems.columns:
    years |= set(pd.to_numeric(linked_cems["report_year"], errors="coerce").dropna().astype(int).tolist())

site_ids = set()
for df in [linked_emissions, linked_cems, linked_catalog]:
    if not df.empty and "site_id" in df.columns:
        site_ids |= set(df["site_id"].dropna().astype(str).tolist())

rows = []
for site_id in sorted(site_ids):
    for year in sorted(years):
        e = linked_emissions[(linked_emissions["site_id"].astype(str) == site_id) & (pd.to_numeric(linked_emissions["report_year"], errors="coerce") == year)] if not linked_emissions.empty else pd.DataFrame()
        c = linked_cems[(linked_cems["site_id"].astype(str) == site_id) & (pd.to_numeric(linked_cems["report_year"], errors="coerce") == year)] if not linked_cems.empty else pd.DataFrame()
        k = linked_catalog[(linked_catalog["site_id"].astype(str) == site_id)] if not linked_catalog.empty else pd.DataFrame()

        emissions_rows = len(e)
        exceedance_rows = int(pd.to_numeric(e.get("limit_exceedance_flag", 0), errors="coerce").fillna(0).sum()) if emissions_rows else 0
        cems_rows = len(c)
        noncomp_rows = int(pd.to_numeric(c.get("noncompliance_events", 0), errors="coerce").fillna(0).sum()) if cems_rows else 0
        report_count = len(k)

        documentary_score = (
            min(emissions_rows, 20) * 0.1
            + min(exceedance_rows, 10) * 0.5
            + min(cems_rows, 20) * 0.1
            + min(noncomp_rows, 10) * 0.5
            + min(report_count, 10) * 0.1
        )

        rows.append({
            "site_id": site_id,
            "report_year": year,
            "documentary_emissions_rows": emissions_rows,
            "documentary_exceedance_rows": exceedance_rows,
            "documentary_cems_rows": cems_rows,
            "documentary_noncompliance_rows": noncomp_rows,
            "documentary_report_count": report_count,
            "documentary_evidence_score": round(float(documentary_score), 3),
        })

fused = pd.DataFrame(rows)
fused.to_csv(out / "documentary_evidence_site_year.csv", index=False)
write_json(out / "documentary_evidence_summary.json", {
    "rows": int(len(fused)),
    "sites": int(fused["site_id"].nunique()) if not fused.empty else 0,
    "years": int(fused["report_year"].nunique()) if not fused.empty else 0,
})

manifest = manifest_base(step, [CONFIGS / "documentary_sources.yml"])
add_artifact(manifest, out / "documentary_evidence_site_year.csv")
add_artifact(manifest, out / "documentary_evidence_summary.json")
write_json(out / "manifest.json", manifest)

display(fused.head(20))